# Chapitre 3 — Données et chunking

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahouahounko/rag-en-pratique/blob/main/chapters/chapitre-03-donnees-et-chunking/03_donnees_et_chunking.ipynb)

Ce laboratoire transforme les 16 extraits du chapitre en expériences exécutables. Les exemples 1 à 13, 15 et 16 sont locaux. Seul l'exemple 14 peut appeler OpenAI.

## Objectifs pédagogiques

À la fin du notebook, vous saurez mesurer un budget de tokens, comparer plusieurs découpages, préserver la structure, enrichir les métadonnées et évaluer les réglages.

## 0. Préparer Colab ou Jupyter

Cette cellule installe uniquement les outils de chunking.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not Path("src").is_dir():
    if not Path("rag-en-pratique").is_dir():
        subprocess.run(["git", "clone", "https://github.com/Ahouahounko/rag-en-pratique.git"], check=True)
    os.chdir("rag-en-pratique")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[chunking]"],
    check=True,
)
print("Environnement du chapitre 3 prêt :", Path.cwd())


## Document fil rouge

Chaque script contient son propre petit jeu de données afin de pouvoir aussi être lancé séparément.

## 1. Compter les tokens

Comparer caractères et tokens avant tout découpage.

Script correspondant : [`01_comptage_tokens.py`](examples/01_comptage_tokens.py)

In [ ]:
# ruff: noqa: F811
"""Mesurer la taille en tokens et brancher ce compteur dans un splitter."""

import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

ENCODER = tiktoken.get_encoding("cl100k_base")


def compter_tokens(texte: str) -> int:
    """Retourne le nombre de tokens produits par l'encodage choisi."""

    return len(ENCODER.encode(texte))


def creer_splitter() -> RecursiveCharacterTextSplitter:
    return RecursiveCharacterTextSplitter(
        chunk_size=640,
        chunk_overlap=64,
        length_function=compter_tokens,
        separators=["\n\n", "\n", ". ", " ", ""],
    )


def main() -> None:
    anglais = "The retrieval system returns relevant documents."
    francais = "Le système de récupération renvoie les documents pertinents."
    print("Anglais :", compter_tokens(anglais), "tokens")
    print("Français :", compter_tokens(francais), "tokens")
    print("Splitter prêt :", creer_splitter())


if __name__ == "__main__":
    main()


## 2. Valider les tailles

Détecter les chunks qui dépasseraient le budget.

Script correspondant : [`02_validation_taille.py`](examples/02_validation_taille.py)

In [ ]:
# ruff: noqa: F811
"""Valider les tailles avant d'appeler un modèle d'embedding."""

import logging

import tiktoken

MAX_TOKENS = 512
MARGE = 0.90
ENCODER = tiktoken.get_encoding("cl100k_base")
logger = logging.getLogger(__name__)


def compter_tokens(texte: str) -> int:
    return len(ENCODER.encode(texte))


def valider_chunks(chunks: list[str]) -> tuple[list[str], list[str]]:
    """Sépare les chunks conformes de ceux qui doivent être redécoupés."""

    conformes: list[str] = []
    a_redecouper: list[str] = []
    seuil = int(MAX_TOKENS * MARGE)
    for chunk in chunks:
        nombre = compter_tokens(chunk)
        if nombre > seuil:
            logger.warning("Chunk de %d tokens > seuil %d", nombre, seuil)
            a_redecouper.append(chunk)
        else:
            conformes.append(chunk)
    return conformes, a_redecouper


if __name__ == "__main__":
    petits, grands = valider_chunks(["Un chunk court.", "mot " * 600])
    print(f"Conformes : {len(petits)} — à redécouper : {len(grands)}")


## 3. Fusionner les chevauchements

Reconstruire un passage sans répéter l'overlap.

Script correspondant : [`03_fusion_overlap.py`](examples/03_fusion_overlap.py)

In [ ]:
# ruff: noqa: F811
def fusionner_chunks_voisins(chunks: list[dict]) -> str:
    """
    Recolle en un passage continu les chunks qui se chevauchent.

    Chaque chunk porte ses offsets dans le document source
    (start_pos / end_pos) : c'est ce qui rend la fusion possible.
    Sans ces metadonnees, on ne peut que dedupliquer approximativement.
    """
    if not chunks:
        return ""

    # 1. Remettre les fragments dans l'ordre du document d'origine
    #    (le retrieval les a renvoyes par score, pas par position)
    ordonnes = sorted(chunks, key=lambda c: c["start_pos"])

    morceaux = [ordonnes[0]["text"]]
    fin_courante = ordonnes[0]["end_pos"]

    for chunk in ordonnes[1:]:
        if chunk["start_pos"] < fin_courante:
            # Chevauchement : on n'ajoute que la partie inedite
            debut_utile = fin_courante - chunk["start_pos"]
            morceaux.append(chunk["text"][debut_utile:])
        else:
            # Chunks disjoints : marqueur de discontinuite explicite
            morceaux.append("\n[...]\n" + chunk["text"])

        fin_courante = max(fin_courante, chunk["end_pos"])

    return " ".join(morceaux)


if __name__ == "__main__":
    exemple = [
        {"text": "Le délai est de trente jours.", "start_pos": 0, "end_pos": 30},
        {"text": "jours. Le remboursement suit.", "start_pos": 24, "end_pos": 53},
    ]
    print(fusionner_chunks_voisins(exemple))


## 4. Ligne de base fixe

Établir une référence simple avant les stratégies avancées.

Script correspondant : [`04_chunking_fixe.py`](examples/04_chunking_fixe.py)

In [ ]:
# ruff: noqa: F811
"""Découpage à taille fixe : la ligne de base à mesurer."""

import tiktoken

ENCODER = tiktoken.get_encoding("cl100k_base")


def decoupage_fixe(texte: str, taille: int = 512, overlap: int = 50) -> list[str]:
    if taille <= 0 or overlap < 0 or overlap >= taille:
        raise ValueError("Il faut taille > 0 et 0 <= overlap < taille")
    tokens = ENCODER.encode(texte)
    pas = taille - overlap
    chunks: list[str] = []
    for debut in range(0, len(tokens), pas):
        fenetre = tokens[debut : debut + taille]
        chunks.append(ENCODER.decode(fenetre))
        if debut + taille >= len(tokens):
            break
    return chunks


if __name__ == "__main__":
    texte = "Le RAG relie une question à des documents pertinents. " * 20
    for index, chunk in enumerate(decoupage_fixe(texte, taille=40, overlap=8), start=1):
        print(index, len(ENCODER.encode(chunk)), "tokens", repr(chunk[:55]))


## 5. Comprendre la récursion

Passer progressivement des grandes frontières aux petites.

Script correspondant : [`05_mecanisme_recursif.py`](examples/05_mecanisme_recursif.py)

In [ ]:
# ruff: noqa: F811
"""Version exécutable du mécanisme récursif présenté en pseudo-code."""


def decouper_recursivement(
    texte: str,
    separateurs: list[str],
    budget: int,
) -> list[str]:
    """Découpe du séparateur le plus structurant au moins structurant."""

    if len(texte) <= budget:
        return [texte.strip()] if texte.strip() else []
    if not separateurs:
        return [texte[index : index + budget] for index in range(0, len(texte), budget)]

    separateur, *restants = separateurs
    morceaux = texte.split(separateur) if separateur else list(texte)
    resultat: list[str] = []
    courant = ""
    for morceau in morceaux:
        candidat = f"{courant}{separateur if courant else ''}{morceau}"
        if len(candidat) <= budget:
            courant = candidat
            continue
        if courant:
            resultat.append(courant.strip())
        if len(morceau) <= budget:
            courant = morceau
        else:
            resultat.extend(decouper_recursivement(morceau, restants, budget))
            courant = ""
    if courant.strip():
        resultat.append(courant.strip())
    return resultat


if __name__ == "__main__":
    document = "Titre\n\nPremier paragraphe assez long.\n\nDeuxième paragraphe détaillé."
    print(decouper_recursivement(document, ["\n\n", ". ", " ", ""], budget=35))


## 6. Respecter Markdown

Prioriser sections, paragraphes, phrases puis mots.

Script correspondant : [`06_chunking_recursif.py`](examples/06_chunking_recursif.py)

In [ ]:
# ruff: noqa: F811
"""Découpage récursif qui privilégie les frontières Markdown."""

import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter

ENCODER = tiktoken.get_encoding("cl100k_base")


def compter_tokens(texte: str) -> int:
    return len(ENCODER.encode(texte))


def decoupage_recursif(document: str, taille: int = 120, overlap: int = 12) -> list[str]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=taille,
        chunk_overlap=overlap,
        length_function=compter_tokens,
        separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_text(document)


if __name__ == "__main__":
    document = "# Guide\n\n## Retours\n\n" + "Retour accepté sous 30 jours. " * 15
    document += "\n## Livraison\n\n" + "Livraison en trois à cinq jours. " * 15
    for index, chunk in enumerate(decoupage_recursif(document), start=1):
        print(f"Chunk {index} ({compter_tokens(chunk)} tokens) :", repr(chunk[:70]))


## 7. Cartographier le document

Conserver le chemin hiérarchique de chaque section.

Script correspondant : [`07_cartographie.py`](examples/07_cartographie.py)

In [ ]:
# ruff: noqa: F811
import re
from dataclasses import dataclass


@dataclass
class SectionDocument:
    """Une zone thematiquement coherente : la future cloison etanche."""
    titre: str
    contenu: str
    chemin: str        # Ex. : "Introduction > Analyse > Risques"
    debut_char: int
    fin_char: int


def cartographier_markdown(texte: str) -> list[SectionDocument]:
    """
    Reconstruit l'arborescence d'un document Markdown a partir
    de ses titres. Le chemin hierarchique complet est conserve :
    c'est lui qui servira de contexte a la phase 3.
    """
    motif_titre = re.compile(r"^(#{1,6})\s+(.+)$", re.MULTILINE)
    occurrences = list(motif_titre.finditer(texte))

    sections, pile = [], []      # pile = chemin hierarchique courant

    for i, occ in enumerate(occurrences):
        niveau = len(occ.group(1))
        titre  = occ.group(2).strip()

        # On tronque la pile au niveau du titre courant, puis on empile
        pile = pile[: niveau - 1]
        pile.append(titre)

        debut = occ.end()
        fin   = occurrences[i + 1].start() if i + 1 < len(occurrences) else len(texte)

        sections.append(SectionDocument(
            titre      = titre,
            contenu    = texte[debut:fin].strip(),
            chemin     = " > ".join(pile),
            debut_char = debut,
            fin_char   = fin,
        ))

    return sections


if __name__ == "__main__":
    markdown = """# Manuel
Introduction générale.
## Paiement
Les cartes sont acceptées.
### Échecs
Vérifiez le plafond de la carte.
## Livraison
Le colis est suivi.
"""
    for section in cartographier_markdown(markdown):
        print(section.chemin, "->", section.contenu)


## 8. Ajouter le contexte

Distinguer le texte vectorisé des métadonnées stockées.

Script correspondant : [`08_context_prepending.py`](examples/08_context_prepending.py)

In [ ]:
# ruff: noqa: F811
from dataclasses import dataclass


@dataclass(frozen=True)          # frozen : l'objet ne change plus apres creation
class ChunkEnrichi:
    """
    L'unite finale, prete a vectoriser.
    Deux sorties distinctes : ce qu'on embarque, ce qu'on stocke.
    """
    texte: str
    source: str
    chemin: str            # "Section: Echecs de paiement > Cartes refusees"
    index: int
    page: int
    debut_char: int
    fin_char: int
    langue: str = "fr"

    def texte_a_vectoriser(self) -> str:
        """
        Context Prepending : l'en-tete hierarchique precede le texte.
        C'est CETTE chaine qui part au modele d'embedding, pas self.texte.
        """
        entete = f"Document: {self.source} > {self.chemin}"
        return f"{entete}\n\n{self.texte}"

    def metadonnees(self) -> dict:
        """
        Stocke a cote du vecteur, jamais vectorise.
        Sert au filtrage prealable, a la citation et a la fusion.
        """
        return {
            "source":     self.source,
            "chemin":     self.chemin,
            "index":      self.index,
            "page":       self.page,
            "debut_char": self.debut_char,
            "fin_char":   self.fin_char,
            "langue":     self.langue,
        }


if __name__ == "__main__":
    chunk = ChunkEnrichi(
        texte="Vérifiez le plafond de la carte.",
        source="manuel_paiement.md",
        chemin="Paiement > Cartes refusées",
        index=0,
        page=3,
        debut_char=120,
        fin_char=154,
    )
    print(chunk.texte_a_vectoriser())
    print(chunk.metadonnees())


## 9. Découper du code Python

Préserver fonctions, classes, imports et numéros de ligne.

Script correspondant : [`09_chunking_ast.py`](examples/09_chunking_ast.py)

In [ ]:
# ruff: noqa: F811
import ast


def decouper_code_python(source: str, chemin: str) -> list[dict]:
    """
    Un chunk = une fonction ou une classe complete,
    prefixee des imports du fichier (contexte de dependance).
    """
    arbre  = ast.parse(source)
    lignes = source.splitlines()

    # 1. Extraire le bloc d'imports : il sera prepende a chaque chunk
    imports = [
        ast.get_source_segment(source, n)
        for n in arbre.body
        if isinstance(n, (ast.Import, ast.ImportFrom))
    ]
    entete_imports = "\n".join(i for i in imports if i)

    # 2. Un chunk par definition de haut niveau
    chunks = []
    for n in arbre.body:
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            corps = "\n".join(lignes[n.lineno - 1 : n.end_lineno])
            chunks.append({
                "texte": f"{entete_imports}\n\n{corps}",
                "metadonnees": {
                    "fichier":     chemin,
                    "symbole":     n.name,
                    "type":        type(n).__name__,
                    "ligne_debut": n.lineno,
                    "ligne_fin":   n.end_lineno,
                    "docstring":   ast.get_docstring(n) or "",
                },
            })

    return chunks


if __name__ == "__main__":
    source = '''import math

def aire_cercle(rayon: float) -> float:
    """Calcule l'aire d'un cercle."""
    return math.pi * rayon ** 2

class Client:
    def __init__(self, nom: str) -> None:
        self.nom = nom
'''
    for chunk in decouper_code_python(source, "geometrie.py"):
        print(chunk["metadonnees"])
        print(chunk["texte"], "\n")


## 10. Préserver les tableaux

Répéter l'en-tête pour que chaque fragment reste lisible.

Script correspondant : [`10_chunking_tableaux.py`](examples/10_chunking_tableaux.py)

In [ ]:
# ruff: noqa: F811
"""Découpage d'un tableau Markdown avec Chonkie."""

from chonkie import TableChunker

TABLEAU_MARKDOWN = """
| Client   | Montant | Statut   |
|----------|---------|----------|
| Dupont   | 15000   | Réglé    |
| Martin   | 8200    | En cours |
| Mensah   | 9100    | Réglé    |
| Diallo   | 7300    | En cours |
"""


def decouper_tableau(tableau: str, lignes_par_chunk: int = 2) -> list[str]:
    """Retourne des fragments dont chacun conserve l'en-tête du tableau."""

    chunker = TableChunker(chunk_size=lignes_par_chunk)
    return [chunk.text for chunk in chunker(tableau)]


if __name__ == "__main__":
    for index, chunk in enumerate(decouper_tableau(TABLEAU_MARKDOWN), start=1):
        print(f"--- Fragment {index} ---\n{chunk}")


## 11. Détecter les ruptures

Utiliser un embedding TF-IDF local et explicable.

Script correspondant : [`11_chunking_semantique.py`](examples/11_chunking_semantique.py)

In [ ]:
# ruff: noqa: F811
"""Chunking sémantique avec un modèle TF-IDF local et explicable."""

import math
import re
from collections import Counter

import numpy as np


class ModeleTFIDF:
    """Petit modèle d'embedding lexical, sans service ni clé API."""

    def embed_documents(self, textes: list[str]) -> list[list[float]]:
        mots = [re.findall(r"\w+", texte.lower()) for texte in textes]
        vocabulaire = sorted({mot for document in mots for mot in document})
        frequences_documents = Counter(mot for mot in vocabulaire for doc in mots if mot in doc)
        vecteurs: list[list[float]] = []
        for document in mots:
            occurrences = Counter(document)
            vecteurs.append(
                [
                    occurrences[mot] * math.log((1 + len(mots)) / (1 + frequences_documents[mot]))
                    for mot in vocabulaire
                ]
            )
        return vecteurs


def chunking_semantique(
    texte: str,
    modele_embedding: ModeleTFIDF,
    percentile: int = 75,
) -> list[str]:
    phrases = [phrase.strip() for phrase in re.split(r"(?<=[.!?])\s+", texte) if phrase.strip()]
    if len(phrases) < 2:
        return [texte]
    vecteurs = np.array(modele_embedding.embed_documents(phrases), dtype=float)
    normes = np.linalg.norm(vecteurs, axis=1, keepdims=True) + 1e-8
    unitaires = vecteurs / normes
    distances = 1.0 - np.sum(unitaires[1:] * unitaires[:-1], axis=1)
    seuil = float(np.percentile(distances, percentile))

    chunks: list[str] = []
    courant = [phrases[0]]
    for index, distance in enumerate(distances, start=1):
        if distance >= seuil:
            chunks.append(" ".join(courant))
            courant = [phrases[index]]
        else:
            courant.append(phrases[index])
    chunks.append(" ".join(courant))
    return chunks


if __name__ == "__main__":
    texte = (
        "Les retours sont acceptés pendant trente jours. "
        "Un remboursement est alors déclenché. "
        "La livraison express arrive demain. "
        "Le transporteur fournit un numéro de suivi."
    )
    for index, chunk in enumerate(chunking_semantique(texte, ModeleTFIDF()), start=1):
        print(index, chunk)


## 12. Accumuler par cohérence

Comparer chaque segment au centre du chunk courant.

Script correspondant : [`12_semantique_accumulation.py`](examples/12_semantique_accumulation.py)

In [ ]:
# ruff: noqa: F811
"""Détection de rupture sémantique par accumulation."""

import re
from collections import Counter

import numpy as np


def vectoriser(texte: str, vocabulaire: list[str]) -> np.ndarray:
    compte = Counter(re.findall(r"\w+", texte.lower()))
    return np.array([compte[mot] for mot in vocabulaire], dtype=float)


def cosinus(gauche: np.ndarray, droite: np.ndarray) -> float:
    denominateur = np.linalg.norm(gauche) * np.linalg.norm(droite)
    return float(np.dot(gauche, droite) / denominateur) if denominateur else 0.0


def chunking_par_accumulation(segments: list[str], seuil: float = 0.15) -> list[str]:
    if not segments:
        return []
    vocabulaire = sorted({mot for segment in segments for mot in re.findall(r"\w+", segment.lower())})
    chunk_courant = [segments[0]]
    vecteurs_courants = [vectoriser(segments[0], vocabulaire)]
    chunks: list[str] = []
    for segment in segments[1:]:
        vecteur = vectoriser(segment, vocabulaire)
        vecteur_reference = np.mean(vecteurs_courants, axis=0)
        if cosinus(vecteur_reference, vecteur) >= seuil:
            chunk_courant.append(segment)
            vecteurs_courants.append(vecteur)
        else:
            chunks.append(" ".join(chunk_courant))
            chunk_courant = [segment]
            vecteurs_courants = [vecteur]
    chunks.append(" ".join(chunk_courant))
    return chunks


if __name__ == "__main__":
    segments = [
        "Le retour produit reste possible trente jours.",
        "Le remboursement du produit suit le retour.",
        "La livraison express utilise un transporteur.",
    ]
    print(chunking_par_accumulation(segments))


## 13. Indexation Parent-Child

Chercher des enfants précis et restituer leurs parents.

Script correspondant : [`13_parent_child.py`](examples/13_parent_child.py)

In [ ]:
# ruff: noqa: F811
"""Indexation Parent-Child locale et exécutable."""

import math
import re
import uuid
from collections import Counter
from dataclasses import dataclass


def decoupage_mots(texte: str, taille: int, overlap: int) -> list[str]:
    mots = texte.split()
    pas = taille - overlap
    return [" ".join(mots[debut : debut + taille]) for debut in range(0, len(mots), pas)]


def similarite_lexicale(gauche: str, droite: str) -> float:
    mots_gauche = Counter(re.findall(r"\w+", gauche.lower()))
    mots_droite = Counter(re.findall(r"\w+", droite.lower()))
    commun = set(mots_gauche) | set(mots_droite)
    produit = sum(mots_gauche[mot] * mots_droite[mot] for mot in commun)
    norme_gauche = math.sqrt(sum(valeur**2 for valeur in mots_gauche.values()))
    norme_droite = math.sqrt(sum(valeur**2 for valeur in mots_droite.values()))
    return produit / (norme_gauche * norme_droite) if norme_gauche and norme_droite else 0.0


@dataclass(frozen=True)
class Enfant:
    texte: str
    id_parent: str


class MagasinEnfants:
    def __init__(self) -> None:
        self.enfants: list[Enfant] = []

    def ajouter(self, texte: str, id_parent: str) -> None:
        self.enfants.append(Enfant(texte, id_parent))

    def recherche(self, requete: str, k: int = 8) -> list[Enfant]:
        return sorted(
            self.enfants,
            key=lambda enfant: similarite_lexicale(requete, enfant.texte),
            reverse=True,
        )[:k]


def indexer_parent_child(document: str, vector_store: MagasinEnfants, docstore: dict[str, str]) -> None:
    for parent_texte in decoupage_mots(document, taille=50, overlap=10):
        id_parent = str(uuid.uuid4())
        docstore[id_parent] = parent_texte
        for enfant_texte in decoupage_mots(parent_texte, taille=15, overlap=3):
            vector_store.ajouter(enfant_texte, id_parent)


def recuperer_avec_parents(
    requete: str,
    vector_store: MagasinEnfants,
    docstore: dict[str, str],
    k: int = 3,
) -> list[str]:
    enfants = vector_store.recherche(requete, k=k)
    ids_uniques = list(dict.fromkeys(enfant.id_parent for enfant in enfants))
    return [docstore[identifiant] for identifiant in ids_uniques]


if __name__ == "__main__":
    document = ("Les retours sont acceptés pendant trente jours. " * 10) + (
        "La livraison standard prend cinq jours ouvrés. " * 10
    )
    magasin = MagasinEnfants()
    parents: dict[str, str] = {}
    indexer_parent_child(document, magasin, parents)
    print(recuperer_avec_parents("Quel délai pour un retour ?", magasin, parents)[0])


## Configuration facultative pour OpenAI

Laissez la case décochée si vous n'avez pas de clé : cela ne bloque aucun autre exemple.

In [ ]:
# @title Exemple 14 — activer OpenAI seulement si vous avez une clé
UTILISER_OPENAI = False # @param {type:"boolean"}

if UTILISER_OPENAI:
    import os
    import subprocess
    import sys
    from getpass import getpass

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[openai]"],
        check=True,
    )
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY : ")
    if not os.getenv("OPENAI_MODEL"):
        os.environ["OPENAI_MODEL"] = input("OPENAI_MODEL : ").strip()
    print("OpenAI est prêt pour l'exemple 14.")
else:
    print("OpenAI désactivé : tous les autres exemples restent exécutables.")


## 14. Contextual Retrieval

Seul exemple facultatif qui appelle OpenAI.

Script correspondant : [`14_contextual_retrieval.py`](examples/14_contextual_retrieval.py)

In [ ]:
# ruff: noqa: F811
"""Contextual Retrieval avec OpenAI, uniquement lorsque la clé est disponible."""

import os
from typing import Protocol

GABARIT = """Voici un document complet :
<document>{document}</document>

Voici un extrait de ce document :
<extrait>{chunk}</extrait>

Rédige en une à deux phrases le contexte nécessaire pour situer cet extrait
dans le document. Ne réponds que par ces phrases, sans préambule."""


class Generateur(Protocol):
    def generer(self, prompt: str) -> str: ...


class GenerateurOpenAI:
    """Adaptateur minimal pour la Responses API."""

    def __init__(self, model: str | None = None) -> None:
        if not os.getenv("OPENAI_API_KEY"):
            raise RuntimeError("OPENAI_API_KEY n'est pas configurée")
        self.model = model or os.getenv("OPENAI_MODEL")
        if not self.model:
            raise RuntimeError("OPENAI_MODEL n'est pas configuré")
        from openai import OpenAI

        self.client = OpenAI()

    def generer(self, prompt: str) -> str:
        response = self.client.responses.create(model=self.model, input=prompt)
        return response.output_text


def contextualiser(document: str, chunks: list[str], llm: Generateur) -> list[str]:
    """Préfixe chaque chunk d'un contexte généré à partir du document complet."""

    enrichis: list[str] = []
    for chunk in chunks:
        contexte = llm.generer(GABARIT.format(document=document, chunk=chunk))
        enrichis.append(f"{contexte.strip()}\n\n{chunk}")
    return enrichis


if __name__ == "__main__":
    if not os.getenv("OPENAI_API_KEY") or not os.getenv("OPENAI_MODEL"):
        print("Exemple facultatif : configurez OPENAI_API_KEY et OPENAI_MODEL pour l'exécuter.")
    else:
        document = "Politique 2026. Les retours sont acceptés sous trente jours avec un reçu."
        print(contextualiser(document, ["Ils exigent un reçu."], GenerateurOpenAI())[0])


## 15. Late Chunking

Découper les vecteurs après encodage global.

Script correspondant : [`15_late_chunking.py`](examples/15_late_chunking.py)

In [ ]:
# ruff: noqa: F811
"""Late Chunking illustré avec un encodeur long-contexte pédagogique."""

import numpy as np
import tiktoken


class ModeleLongContextePedagogique:
    """Produit un vecteur par token enrichi par la moyenne du document."""

    def __init__(self, dimension: int = 8) -> None:
        self.dimension = dimension
        self.tokenizer = tiktoken.get_encoding("cl100k_base")

    def encode_tokens(self, document: str) -> np.ndarray:
        ids = np.array(self.tokenizer.encode(document), dtype=float)
        dimensions = np.arange(1, self.dimension + 1, dtype=float)
        locaux = np.sin(ids[:, None] / dimensions[None, :])
        contexte_global = locaux.mean(axis=0, keepdims=True)
        return locaux + contexte_global


def late_chunking(
    document: str,
    modele_long_contexte: ModeleLongContextePedagogique,
    frontieres: list[tuple[int, int]],
) -> list[np.ndarray]:
    vecteurs_tokens = modele_long_contexte.encode_tokens(document)
    return [vecteurs_tokens[debut:fin].mean(axis=0) for debut, fin in frontieres]


def frontieres_en_tokens(
    document: str,
    chunk_size_chars: int,
    tokenizer: tiktoken.Encoding,
) -> list[tuple[int, int]]:
    frontieres: list[tuple[int, int]] = []
    for debut_caractere in range(0, len(document), chunk_size_chars):
        fin_caractere = min(debut_caractere + chunk_size_chars, len(document))
        debut_token = len(tokenizer.encode(document[:debut_caractere]))
        fin_token = len(tokenizer.encode(document[:fin_caractere]))
        if fin_token > debut_token:
            frontieres.append((debut_token, fin_token))
    return frontieres


if __name__ == "__main__":
    document = (
        "Les retours sont acceptés sous trente jours. "
        "La livraison standard prend cinq jours ouvrés."
    )
    modele = ModeleLongContextePedagogique()
    frontieres = frontieres_en_tokens(document, 45, modele.tokenizer)
    vecteurs = late_chunking(document, modele, frontieres)
    for frontiere, vecteur in zip(frontieres, vecteurs, strict=True):
        texte = modele.tokenizer.decode(modele.tokenizer.encode(document)[slice(*frontiere)])
        print(frontiere, repr(texte), "->", np.round(vecteur[:3], 3))


## 16. Comparer les réglages

Évaluer plusieurs tailles, overlaps et stratégies.

Script correspondant : [`16_grid_search.py`](examples/16_grid_search.py)

In [ ]:
# ruff: noqa: F811
"""Explorer systématiquement taille, overlap et stratégie de chunking."""

import itertools
import re

TAILLES = [12, 20]
OVERLAPS = [0.0, 0.20]
STRATEGIES = ["fixe", "paragraphes"]


def decouper(document: str, taille: int, overlap: int, strategie: str) -> list[str]:
    if strategie == "paragraphes":
        return [partie.strip() for partie in document.split("\n\n") if partie.strip()]
    mots = document.split()
    pas = max(1, taille - overlap)
    return [" ".join(mots[index : index + taille]) for index in range(0, len(mots), pas)]


def score_lexical(question: str, chunk: str) -> int:
    mots_question = set(re.findall(r"\w+", question.lower()))
    mots_chunk = set(re.findall(r"\w+", chunk.lower()))
    return len(mots_question & mots_chunk)


def evaluer(questions: list[tuple[str, str]], chunks: list[str]) -> dict[str, float]:
    succes = 0
    for question, terme_attendu in questions:
        meilleur = max(chunks, key=lambda chunk: score_lexical(question, chunk))
        succes += terme_attendu.lower() in meilleur.lower()
    return {"precision_contextuelle": succes / len(questions)}


def explorer(corpus: list[str], questions: list[tuple[str, str]]) -> list[dict[str, object]]:
    resultats: list[dict[str, object]] = []
    for taille, ratio, strategie in itertools.product(TAILLES, OVERLAPS, STRATEGIES):
        chunks = [
            chunk
            for document in corpus
            for chunk in decouper(document, taille, int(taille * ratio), strategie)
        ]
        resultats.append(
            {
                "taille": taille,
                "overlap": ratio,
                "strategie": strategie,
                "nombre_chunks": len(chunks),
                **evaluer(questions, chunks),
            }
        )
    return resultats


if __name__ == "__main__":
    corpus = [
        "Retours\n\nLes produits sont retournables pendant trente jours avec un reçu.",
        "Livraison\n\nLa livraison standard prend cinq jours ouvrés avec suivi.",
    ]
    questions = [("Quel délai pour un retour ?", "trente"), ("Délai livraison ?", "cinq")]
    resultats = explorer(corpus, questions)
    for resultat in resultats:
        print(resultat)
    print("Meilleure configuration :", max(resultats, key=lambda r: r["precision_contextuelle"]))


## Bilan

Il n'existe pas de taille universelle. Mesurez les tokens, conservez les métadonnées de position, puis comparez les stratégies sur des questions représentatives de votre corpus.